# Notebook 05 (Google Colab) — Embeddings y Transformers

Versión para **Google Colab con GPU** del Notebook 05 local (`05_embeddings_and_transformers.ipynb`). Corré aquí la **Parte A** (GloVe + Word2Vec) y la **Parte B** (fine-tuning de DistilBERT). El resto del pipeline (01–04, 06, 07) se ejecuta en local.

## Objetivo

Comparar modelos más complejos sobre el **subconjunto político**:
- **Parte A:** Embeddings estáticos (GloVe preentrenado + Word2Vec de dominio) + LR/SVM
- **Parte B:** Fine-tuning de DistilBERT con grilla de hiperparámetros

**Requisitos previos (local)**
1. Correr los notebooks `01 → 02 → 03 → 04` (y opcionalmente `06`).
2. Subir a una carpeta de tu **Google Drive** los *inputs* (gitignoreados; los genera el pipeline local):
   - `data/processed/politics_train.parquet`, `politics_val.parquet`, `politics_test.parquet`
   - `results/metrics/source_ablation_decision.json`
3. Tener la rama `development` del repo pusheada (este notebook la clona y hace checkout).
4. Runtime con GPU: **Runtime → Change runtime type → T4 GPU**.

Al final descargás un `.zip` con los artefactos (CSVs + figuras + el modelo `best_distilbert/`) y los importás en tu repo local siguiendo la **sección 9** de este notebook.

> Los modelos aprenden patrones lingüísticos del dataset; no verifican hechos.

## 0. Verificar GPU

In [ ]:
import subprocess, torch

out = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(out or "Sin GPU. Activá Runtime > Change runtime type > T4 GPU y reconectá.")
print("CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Clonar el repo y hacer checkout de `development`

In [ ]:
import os, sys

REPO_URL = "https://github.com/gcandisano/nlp.git"
if not os.path.exists("/content/nlp"):
    !git clone {REPO_URL} /content/nlp
%cd /content/nlp
!git checkout development
!git log --oneline -1
if "/content/nlp/src" not in sys.path:
    sys.path.insert(0, "/content/nlp/src")
print("cwd:", os.getcwd())

## 2. Instalar dependencias

Solo lo que necesita el Exp 3: `transformers`/`accelerate` (DistilBERT) y `gensim` (GloVe/Word2Vec). No hace falta spaCy/NLTK aquí. Si Colab pide reiniciar el runtime por numpy/scipy, reiniciá y re-ejecutá desde la celda 1.

In [ ]:
!pip install -q -U "transformers>=4.41" accelerate gensim pyarrow

## 3. Inputs desde Google Drive

Ajustá `DRIVE_INPUTS` a la carpeta de tu Drive donde subiste los 4 archivos.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# >>> Ajustá a tu carpeta de Drive <<<
DRIVE_INPUTS = "/content/drive/MyDrive/nlp_exp3_inputs"

import os, shutil
os.makedirs("data/processed", exist_ok=True)
os.makedirs("results/metrics", exist_ok=True)
INPUTS = {
    "politics_train.parquet": "data/processed/politics_train.parquet",
    "politics_val.parquet":   "data/processed/politics_val.parquet",
    "politics_test.parquet":  "data/processed/politics_test.parquet",
    "source_ablation_decision.json": "results/metrics/source_ablation_decision.json",
}
for name, dst in INPUTS.items():
    src = os.path.join(DRIVE_INPUTS, name)
    assert os.path.exists(src), f"Falta en Drive: {src}"
    shutil.copy(src, dst)
    print("OK", dst)

## 4. Imports y decisión de ablación de fuente

Leemos `source_ablation_decision.json` directamente (sin importar `nlp.features`) para no requerir spaCy/VADER en Colab.

In [ ]:
import json
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, fbeta_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

from nlp.io import load_splits
from nlp.metrics import compute_metrics, metrics_to_row
from nlp.paths import (
    DATA_EMBEDDINGS,
    RANDOM_STATE,
    RESULTS_FIGURES,
    RESULTS_METRICS,
    RESULTS_MODELS,
    ensure_output_dirs,
    word2vec_model_path,
)
from nlp.embeddings import (
    embedding_text_series,
    load_glove_vectors,
    load_or_compute_document_embeddings,
    load_or_compute_glove_embeddings,
    train_or_load_word2vec,
    tune_and_evaluate_dense_classifiers,
)
from nlp.transformers_data import NewsDataset, prepare_transformer_texts

ensure_output_dirs()
sns.set_theme(style="whitegrid")

with open(RESULTS_METRICS / "source_ablation_decision.json") as fh:
    source_decision = json.load(fh)
normalize_source = bool(source_decision.get("use_source_normalization", False))
print("normalize_source =", normalize_source, "| f2_drop =", source_decision.get("f2_drop"))

## 5. Cargar splits políticos

In [ ]:
EMBEDDING_COLS = ["label", "title", "text", "clean_full_text_without_stopwords"]
pol_train, pol_val, pol_test = load_splits("politics", columns=EMBEDDING_COLS)

TEXT_COL = "clean_full_text_without_stopwords"
EMB_DIM = 100
CACHE_PREFIX = "politics"

train_texts = embedding_text_series(pol_train, TEXT_COL, normalize_source=normalize_source)
val_texts = embedding_text_series(pol_val, TEXT_COL, normalize_source=normalize_source)
test_texts = embedding_text_series(pol_test, TEXT_COL, normalize_source=normalize_source)
y_train, y_val, y_test = pol_train["label"], pol_val["label"], pol_test["label"]
print("train/val/test:", len(pol_train), len(pol_val), len(pol_test))

## 6. Parte A — Embeddings estáticos (GloVe + Word2Vec)

GloVe (`glove.6B.100d`, ~850 MB) se descarga la primera vez. Representación = promedio de vectores; clasificador LR/LinearSVC con `StandardScaler`; selección de C por F2 en validación.

In [ ]:
glove = load_glove_vectors(EMB_DIM)
print(f"GloVe: {len(glove):,} vectores")
w2v = train_or_load_word2vec(
    train_texts, word2vec_model_path(CACHE_PREFIX, EMB_DIM), vector_size=EMB_DIM
).wv
print(f"Word2Vec: {len(w2v):,} vectores (entrenado en train politics)")

Xtr_g = load_or_compute_glove_embeddings(train_texts, DATA_EMBEDDINGS / f"{CACHE_PREFIX}_train_glove{EMB_DIM}d.npz", glove, EMB_DIM)
Xva_g = load_or_compute_glove_embeddings(val_texts,   DATA_EMBEDDINGS / f"{CACHE_PREFIX}_val_glove{EMB_DIM}d.npz",   glove, EMB_DIM)
Xte_g = load_or_compute_glove_embeddings(test_texts,  DATA_EMBEDDINGS / f"{CACHE_PREFIX}_test_glove{EMB_DIM}d.npz",  glove, EMB_DIM)
Xtr_w = load_or_compute_document_embeddings(train_texts, DATA_EMBEDDINGS / f"{CACHE_PREFIX}_train_w2v{EMB_DIM}d.npz", w2v, EMB_DIM, tag="word2vec")
Xva_w = load_or_compute_document_embeddings(val_texts,   DATA_EMBEDDINGS / f"{CACHE_PREFIX}_val_w2v{EMB_DIM}d.npz",   w2v, EMB_DIM, tag="word2vec")
Xte_w = load_or_compute_document_embeddings(test_texts,  DATA_EMBEDDINGS / f"{CACHE_PREFIX}_test_w2v{EMB_DIM}d.npz",  w2v, EMB_DIM, tag="word2vec")

embedding_results = []
embedding_results += tune_and_evaluate_dense_classifiers(
    Xtr_g, y_train, Xva_g, y_val, Xte_g, y_test,
    representation="glove_avg", use_source_normalization=normalize_source,
)
embedding_results += tune_and_evaluate_dense_classifiers(
    Xtr_w, y_train, Xva_w, y_val, Xte_w, y_test,
    representation="word2vec_avg", use_source_normalization=normalize_source,
)
embedding_df = pd.DataFrame(embedding_results)
embedding_df.to_csv(RESULTS_METRICS / "embedding_results.csv", index=False)
embedding_df

In [ ]:
plot_emb = embedding_df.copy()
plot_emb["modelo"] = plot_emb["representation"] + " / " + plot_emb["model"]
plot_emb = plot_emb.sort_values("f2_fake")
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=plot_emb, x="f2_fake", y="modelo", hue="representation", dodge=False, ax=ax)
ax.set_xlabel("F2 fake (test)"); ax.set_ylabel(""); ax.set_title("Embeddings: GloVe vs Word2Vec")
fig.savefig(RESULTS_FIGURES / "05_embedding_comparison.png", bbox_inches="tight", dpi=150)
plt.show()

## 7. Parte B — Fine-tuning de DistilBERT (config para T4)

**Datos completos** (`SAMPLE_FRAC=1.0`, no DEV). Grilla de HP reducida pero real (6 combos, ~40–60 min en T4), buscando sobre learning rate y warmup. Para la grilla completa del Informe, ampliá `BATCH_SIZES`/`EPOCHS_LIST` (≈24 combos, varias horas).

In [ ]:
SAMPLE_FRAC = 1.0  # datos COMPLETOS — no DEV
LEARNING_RATES = [2e-5, 3e-5, 5e-5]
BATCH_SIZES = [16]        # grilla completa del Informe: [8, 16]
EPOCHS_LIST = [3]         # grilla completa del Informe: [2, 3]
WARMUP_RATIOS = [0.0, 0.1]
MAX_LENGTHS = [256]
MAX_COMBOS = None
EARLY_STOPPING_PATIENCE = 1
CHECKPOINT_DIR = RESULTS_MODELS / "distilbert_checkpoints"

assert SAMPLE_FRAC == 1.0, "Entrenar con datos completos para resultados comparables."
n_combos = len(LEARNING_RATES) * len(BATCH_SIZES) * len(EPOCHS_LIST) * len(WARMUP_RATIOS) * len(MAX_LENGTHS)
print("Combinaciones de HP a explorar:", n_combos)

In [ ]:
tr, va, te = pol_train, pol_val, pol_test
tr_texts = prepare_transformer_texts(embedding_text_series(tr, TEXT_COL, normalize_source=normalize_source))
va_texts = prepare_transformer_texts(embedding_text_series(va, TEXT_COL, normalize_source=normalize_source))
te_texts = prepare_transformer_texts(embedding_text_series(te, TEXT_COL, normalize_source=normalize_source))

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def compute_transformer_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    _, _, f1, _ = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
    return {
        "f2_fake": fbeta_score(labels, preds, beta=2, pos_label=1, zero_division=0),
        "f1_fake": f1[1] if len(f1) > 1 else 0.0,
        "accuracy": accuracy_score(labels, preds),
    }

In [ ]:
use_fp16 = torch.cuda.is_available()
combo_grid = list(product(LEARNING_RATES, BATCH_SIZES, EPOCHS_LIST, WARMUP_RATIOS, MAX_LENGTHS))
if MAX_COMBOS is not None:
    combo_grid = combo_grid[:MAX_COMBOS]

val_rows, best_val_f2, best_bundle = [], -1.0, None
for lr, bs, epochs, warmup_ratio, max_len in combo_grid:
    print(f"\n=== lr={lr}, bs={bs}, epochs={epochs}, warmup={warmup_ratio}, max_len={max_len} ===")
    train_ds = NewsDataset(tr_texts, tr["label"], tokenizer, max_len)
    val_ds = NewsDataset(va_texts, va["label"], tokenizer, max_len)
    model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
    args = TrainingArguments(
        output_dir=str(CHECKPOINT_DIR / f"lr{lr}_bs{bs}_ep{epochs}_wu{warmup_ratio}_len{max_len}"),
        learning_rate=lr,
        per_device_train_batch_size=bs,
        per_device_eval_batch_size=bs * 2,
        num_train_epochs=epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f2_fake",
        greater_is_better=True,
        warmup_ratio=warmup_ratio,
        lr_scheduler_type="linear",
        logging_steps=50,
        seed=RANDOM_STATE,
        report_to="none",
        dataloader_num_workers=2,
        fp16=use_fp16,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
        compute_metrics=compute_transformer_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    trainer.train()

    val_pred = trainer.predict(val_ds)
    y_val_pred = np.argmax(val_pred.predictions, axis=-1)
    vm = compute_metrics(va["label"], y_val_pred)
    val_rows.append({
        "learning_rate": lr, "batch_size": bs, "epochs": epochs,
        "warmup_ratio": warmup_ratio, "max_length": max_len, "sample_frac": SAMPLE_FRAC, **vm,
    })
    print(f"F2 val: {vm['f2_fake']:.4f}")

    if vm["f2_fake"] > best_val_f2:
        best_val_f2 = vm["f2_fake"]
        best_bundle = {
            "model": trainer.model, "max_len": max_len, "lr": lr,
            "bs": bs, "epochs": epochs, "warmup_ratio": warmup_ratio,
        }

val_search_df = pd.DataFrame(val_rows)
val_search_df.to_csv(RESULTS_METRICS / "transformer_val_search.csv", index=False)
val_search_df.sort_values("f2_fake", ascending=False)

In [ ]:
test_ds = NewsDataset(te_texts, te["label"], tokenizer, best_bundle["max_len"])
pred = Trainer(model=best_bundle["model"], data_collator=data_collator).predict(test_ds)
y_pred = np.argmax(pred.predictions, axis=-1)
y_proba = torch.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy()
test_m = compute_metrics(te["label"], y_pred, y_proba)

transformer_row = metrics_to_row(test_m, {
    "model": "distilbert-base-uncased",
    "learning_rate": best_bundle["lr"],
    "batch_size": best_bundle["bs"],
    "epochs": best_bundle["epochs"],
    "warmup_ratio": best_bundle["warmup_ratio"],
    "max_length": best_bundle["max_len"],
    "sample_frac": SAMPLE_FRAC,
    "dataset_scope": "politics",
    "split": "test",
    "use_source_normalization": normalize_source,
})
transformer_df = pd.DataFrame([transformer_row])
transformer_df.to_csv(RESULTS_METRICS / "transformer_results.csv", index=False)
best_bundle["model"].save_pretrained(str(RESULTS_MODELS / "best_distilbert"))
tokenizer.save_pretrained(str(RESULTS_MODELS / "best_distilbert"))

print("Mejor config (val):", {k: best_bundle[k] for k in ["lr", "bs", "epochs", "warmup_ratio"]})
print("Test:", {k: round(v, 4) for k, v in test_m.items() if k != "roc_auc"})
transformer_df

In [ ]:
plot_val = val_search_df.sort_values("f2_fake", ascending=False).copy()
plot_val["config"] = plot_val.apply(
    lambda r: f"lr={r['learning_rate']}, bs={int(r['batch_size'])}, wu={r['warmup_ratio']}", axis=1
)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=plot_val, x="f2_fake", y="config", color="#4c72b0", ax=ax)
ax.set_xlabel("F2 fake (validación)"); ax.set_ylabel(""); ax.set_title("DistilBERT: búsqueda de hiperparámetros")
fig.savefig(RESULTS_FIGURES / "05_transformer_hp_search.png", bbox_inches="tight", dpi=150)
plt.show()

## 8. Exportar artefactos (a Drive + descarga)

Genera `exp3_outputs.zip` con los CSVs, las figuras y el modelo `best_distilbert/`. Lo copia a tu carpeta de Drive (persistencia) y lo descarga. Importalo en el repo local siguiendo la **sección 9**.

In [ ]:
import zipfile

ARTIFACTS = [
    "results/metrics/embedding_results.csv",
    "results/metrics/transformer_results.csv",
    "results/metrics/transformer_val_search.csv",
    "results/figures/05_embedding_comparison.png",
    "results/figures/05_transformer_hp_search.png",
]
zip_path = "/content/exp3_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in ARTIFACTS:
        if os.path.exists(p):
            z.write(p)
        else:
            print("(falta)", p)
    for root, _, fnames in os.walk("results/models/best_distilbert"):
        for fn in fnames:
            z.write(os.path.join(root, fn))
print("Zip listo:", zip_path)

try:
    shutil.copy(zip_path, os.path.join(DRIVE_INPUTS, "exp3_outputs.zip"))
    print("Copiado a Drive:", os.path.join(DRIVE_INPUTS, "exp3_outputs.zip"))
except Exception as e:
    print("No se pudo copiar a Drive:", e)

from google.colab import files
files.download(zip_path)

## 9. Importar resultados en el repo local

Una vez descargado `exp3_outputs.zip` desde Colab:

### 9.1 Descomprimir en la raíz del repo

```bash
cd /ruta/a/tu/repo/nlp
unzip -o ~/Downloads/exp3_outputs.zip
```

Esto restaura las rutas relativas:
- `results/metrics/embedding_results.csv`
- `results/metrics/transformer_results.csv`
- `results/metrics/transformer_val_search.csv`
- `results/models/best_distilbert/` (carpeta completa: `config.json`, `model.safetensors`, tokenizer, etc.)
- `results/figures/05_*.png`

No hace falta bajar `data/embeddings/` (cache de GloVe/Word2Vec); solo sirve si re-correrías el 05 en Colab.

### 9.2 Verificar que la corrida fue completa (no DEV)

Este notebook usa `SAMPLE_FRAC=1.0` y no activa `NLP_DEV_MODE`. Confirmá en local:

```bash
uv run python -c "import pandas as pd; d=pd.read_csv('results/metrics/transformer_results.csv'); print(d[['model','f2_fake','sample_frac']].to_string(index=False)); assert (d['sample_frac']==1.0).all(), 'CORRIDA DEV: re-correr en Colab'"
```

También verificá que existan (sin sufijo `_dev`):
- `results/metrics/transformer_results.csv`
- `results/models/best_distilbert/`

| Corrida | Archivos generados | ¿Comparable? |
| :------ | :----------------- | :----------- |
| **Full** (correcta) | `transformer_results.csv`, `best_distilbert/` | ✅ sí |
| **DEV** (incorrecta) | `transformer_results_dev.csv`, `best_distilbert_dev/` | ❌ no — re-correr |

### 9.3 Re-consolidar la tabla comparativa

```bash
uv run python -c "from nlp.metrics import consolidate_results; print(consolidate_results()[['model','dataset_scope','f2_fake','split']].to_string(index=False))"
```

Deberías ver filas de `glove_avg`, `word2vec_avg` y `distilbert-base-uncased` (todas `politics`/`test`), además de baselines y el modelo lingüístico.

### 9.4 Re-ejecutar el Notebook 07

```bash
uv run jupyter lab
```

Con `best_distilbert/` y `transformer_results.csv` presentes, el Notebook 07:
- genera predicciones de DistilBERT en test y la plantilla `results/error_analysis/error_annotations_transformer.csv` (para anotar a mano);
- actualiza `results/figures/07_model_comparison_f2.png` (comparación de los 5 modelos);
- permite comparar categorías de error baseline vs. Transformer una vez completas ambas anotaciones.

### Checklist

- [ ] `sample_frac == 1.0` en `transformer_results.csv`
- [ ] Carpeta `best_distilbert/` completa importada
- [ ] `consolidate_results()` incluye embeddings y DistilBERT
- [ ] Notebook 07 re-ejecutado
- [ ] Actualizar en el Informe los números de la tabla comparativa